In [ ]:
# pyright: reportGeneralTypeIssues=false, reportUnknownMemberType=false, reportUnknownVariableType=false, reportUnknownArgumentType=false
# ruff: noqa
# pylint: skip-file

# NHANES Diabetes Prediction - Bayesian Hyperparameter Optimization

This notebook implements Bayesian optimization using Optuna with W&B tracking for the LightGBM diabetes screening model.

**Optimization target**: Recall (prioritizing detection of positive cases for screening purposes)

## 1. Setup and Configuration

In [ ]:
import lightgbm as lgb
import matplotlib.pyplot as plt

import optuna
import pandas as pd
import wandb
from optuna.integration.wandb import WeightsAndBiasesCallback
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    recall_score,
)
from sklearn.model_selection import train_test_split  # pyright: ignore[reportUnknownVariableType]
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    precision_recall_curve,
    auc,
    recall_score,
    precision_score,
    f1_score,
)
from sklearn.metrics import average_precision_score

import numpy as np

In [ ]:
# Configuration

# Weights and Biases
WANDB_PROJECT = "Model exploration for Diabetes Prediction"
ENTITY = "fastegiano-tesis"

# Optuna
STORAGE = "sqlite:///optuna_diabetes.db"
STUDY_NAME = "lgb_ap_10cv_20172021_FE"
PROJECT_NAME = "Model exploration for Diabetes Prediction"
RANDOM_STATE = 37
N_TRIALS = 200

# Post-training options
USE_CALIBRATION = False  # Toggle probability calibration on/off

# Categorical features
CAT_FEATURES = [
    "education_level",
    # "moderate_every_X_days",
    # "vigorous_every_X_days",
    "has_partner",
    "had_partner",
    "is_female",
    "ever_smoker",
    "is_current_smoker",
    "drinking_frequency",
]

## 2. Data Loading and Preparation

In [ ]:
data = pd.read_csv("../../../dataset/processed_data_combined_2017_2021.csv")  # pyright: ignore[reportUnknownMemberType]
data.head()

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
# Drop survey_weight if present (no error if absent) and show new shape
class_weights = data["survey_weight"].value_counts().to_dict()
data.drop(columns="survey_weight", inplace=True, errors="ignore")
print(f"Data shape: {data.shape}")

### Feature Engineering

In [ ]:
data["waist_to_height_ratio"] = data["BMXWAIST"] / data["BMXHT"]
data["age_bmi_interaction"] = data["RIDAGEYR"] * data["BMXBMI"]

In [ ]:
def create_age_bins(
    df: pd.DataFrame,
    age_column: str = "RIDAGEYR",
    age_bins: tuple[int, ...] = (18, 45, 65, 79),
    age_labels: tuple[str, ...] = ("young_adult", "middle_age", "senior"),
    elderly_label: str = "elderly",
    unknown_label: str = "age_unknown",
    elderly_top_coded_age: int = 80,
) -> pd.DataFrame:
    """Bin age into categories and one-hot encode."""
    age = df[age_column].copy()
    age_group = pd.Series(index=df.index, dtype="object")

    missing_mask = age.isna()
    elderly_mask = age >= elderly_top_coded_age

    age_group[elderly_mask] = elderly_label

    valid_mask = ~missing_mask & ~elderly_mask
    age_group[valid_mask] = pd.cut(
        age[valid_mask],
        bins=list(age_bins),
        labels=age_labels,
        right=False,
    )

    age_group[missing_mask] = unknown_label

    dummies = pd.get_dummies(age_group, prefix="age", dtype=int)

    unknown_col = f"age_{unknown_label}"
    if unknown_col in dummies.columns and missing_mask.sum() == 0:
        dummies = dummies.drop(columns=[unknown_col])

    return pd.concat([df, dummies], axis=1)

In [ ]:
data = create_age_bins(data)

### X y Split

In [ ]:
X = data.drop(columns=["has_diabetes_or_prediabetes", "cycle"])
y = data["has_diabetes_or_prediabetes"]

In [ ]:
# First split: separate out the untouchable test set
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.10, stratify=y, random_state=RANDOM_STATE
)

# Second split: carve out validation for threshold tuning
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=0.11,
    stratify=y_trainval,
    random_state=RANDOM_STATE,
)
# 0.11 of 90% ≈ 10% of total → gives you ~80/10/10 split

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

### Null revision

In [ ]:
# Check for null values in training and test sets
print("=" * 70)
print("NULL VALUES SUMMARY")
print("=" * 70)

# Create a summary DataFrame
null_summary = pd.DataFrame({
    "X_train_null_%": (X_train.isnull().sum() / len(X_train)) * 100,
    "X_test_null_%": (X_test.isnull().sum() / len(X_test)) * 100,
})

# Filter to only show columns with any nulls (optional)
null_summary = null_summary[
    (null_summary["X_train_null_%"] > 0) | (null_summary["X_test_null_%"] > 0)
]

# Sort by X_train nulls descending
null_summary = null_summary.sort_values("X_train_null_%", ascending=False)

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(
    f"\nTotal null values - X_train: {X_train.isnull().sum().sum()}, X_test: {X_test.isnull().sum().sum()}"
)

if len(null_summary) > 0:
    print(f"\nColumns with null values:\n")
    print(null_summary.to_string())
else:
    print("\nNo null values found in either dataset!")

print("\n" + "=" * 70)

---

## 3. Model preparation

## Weights & Biases initialization

In [ ]:
WANDB_KWARGS = {  # type: ignore
    "entity": ENTITY,
    "project": WANDB_PROJECT,
    "name": STUDY_NAME,
    "config": {
        "random_state": RANDOM_STATE,
        "n_trials": N_TRIALS,
        "metric": "average_precision",
    },
}

## Optimization set up

In [ ]:
from typing import Any

fixed_params: dict[str, Any] = {
    "objective": "binary",
    "metric": "binary_logloss",
    "verbosity": -1,
    "n_estimators": 2000,
    # "is_unbalance": True,
    "bagging_seed": RANDOM_STATE,
    "feature_fraction_seed": RANDOM_STATE,
    "random_state": RANDOM_STATE,
}

In [ ]:
def objective(trial, fixed_params=fixed_params):  # type: ignore

    # Suggest max_depth first since num_leaves depends on it
    max_depth = trial.suggest_int("max_depth", 3, 8)  # type: ignore

    params = {  # pyright: ignore[reportUnknownVariableType]
        **fixed_params,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),  # type: ignore
        "max_depth": max_depth,  # type: ignore
        "num_leaves": trial.suggest_int("num_leaves", 2, 2**max_depth - 1),  # type: ignore
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 100),  # type: ignore
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.1, 1.0),  # type: ignore
        "bagging_freq": trial.suggest_int("bagging_freq", 3, 10),  # type: ignore
        "feature_fraction": trial.suggest_float("feature_fraction", 0.1, 1.0),  # type: ignore
    }

    kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
    val_scores = []
    train_scores = []
    n_iterations = []  # Track early stopping iterations

    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X_train, y_train)):  # type: ignore
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]  # type: ignore
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]  # type: ignore

        model = lgb.LGBMClassifier(**params)  # type: ignore
        model.fit(  # type: ignore
            X_tr,
            y_tr,  # type: ignore
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(50, verbose=False)],
            categorical_feature=CAT_FEATURES,
        )

        # Track how many iterations before early stopping
        n_iterations.append(model.best_iteration_)

        # Get probabilities (threshold-agnostic)
        y_val_proba = model.predict_proba(X_val)[:, 1]
        y_train_proba = model.predict_proba(X_tr)[:, 1]

        # Average Precision = area under PR curve
        val_ap = average_precision_score(y_val, y_val_proba)
        train_ap = average_precision_score(y_tr, y_train_proba)

        val_scores.append(val_ap)
        train_scores.append(train_ap)

    # Calculate mean metrics across folds
    mean_val_ap = np.mean(val_scores)
    mean_train_ap = np.mean(train_scores)
    overfitting_gap = mean_train_ap - mean_val_ap

    # Log aggregated trial metrics to W&B with explicit step
    wandb.log(
        {
            # Primary metric
            "mean_val_ap": mean_val_ap,
            # Stability
            "std_val_ap": np.std(val_scores),
            "min_fold_ap": np.min(val_scores),
            # Overfitting
            "overfitting_gap": overfitting_gap,
            # Convergence
            "mean_n_iterations": np.mean(n_iterations),
        },
        step=trial.number,
    )

    return mean_val_ap

## 4. Run Optimization

In [ ]:
# Delete all studies
"""
optuna.delete_study(
    study_name=STUDY_NAME,
    storage="sqlite:///optuna.db"  # change if your storage is different
)
"""

In [ ]:
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE,
    load_if_exists=True,
    direction="maximize",
)

In [ ]:
wandb_callback = WeightsAndBiasesCallback(
    metric_name="average_precision",
    wandb_kwargs=WANDB_KWARGS,  # type: ignore
)

study.optimize(objective, n_trials=N_TRIALS, callbacks=[wandb_callback])  # type: ignore

wandb.finish()

## 5. Results Analysis

In [ ]:
# Display best params from the Optuna study
best_params = study.best_params
print("Optimized parameters:")
for k, v in best_params.items():
    print(f"{k}: {v}")

print(f"\nBest AP (study.best_value): {study.best_value:.4f}")

## 6. Best Model Evaluation

In [ ]:
from typing import Any


def get_full_params(
    fixed_params: dict[str, Any], best_params: dict[str, Any]
) -> dict[str, Any]:
    return {**fixed_params, **best_params}

In [ ]:
best_model = lgb.LGBMClassifier(**get_full_params(fixed_params, study.best_params))

best_model.fit(
    X_train,
    y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    eval_names=["train", "valid"],
    callbacks=[lgb.early_stopping(50, verbose=False)],
    categorical_feature=CAT_FEATURES,
)  # type: ignore

y_pred = best_model.predict(X_test)  # type: ignore

### Probability Calibration (Optional)

Controlled by `USE_CALIBRATION` flag. When enabled, calibrates probability estimates using isotonic regression on the validation set.

In [ ]:
# Conditionally calibrate and select inference model
if USE_CALIBRATION:
    calibrated_model = CalibratedClassifierCV(
        best_model, method="isotonic", cv="prefit"
    )
    calibrated_model.fit(X_val, y_val)
    inference_model = calibrated_model
    print("Using CALIBRATED model (isotonic regression on validation set)")
else:
    inference_model = best_model
    print("Using UNCALIBRATED model (raw probabilities)")

In [ ]:
recall_test_positive = np.round(recall_score(y_test, y_pred, pos_label=1), 2)  # type: ignore
recall_test_negative = np.round(recall_score(y_test, y_pred, pos_label=0), 2)  # type: ignore

In [ ]:
print(f"y_test value counts:\n{y_test.value_counts()}")
print(f"y_pred value counts:\n{pd.Series(y_pred).value_counts()}")
print(f"Recall test (pos_label=1): {recall_test_positive}")
print(f"Recall test (pos_label=0): {recall_test_negative}")

### Training Progress - Binary Log Loss

In [ ]:
# Extract evaluation results
results = best_model.evals_result_

# Plot the progression
plt.figure(figsize=(10, 6))  # type: ignore
plt.plot(results["train"]["binary_logloss"], label="Train", linewidth=2)  # type: ignore
plt.plot(results["valid"]["binary_logloss"], label="Valid", linewidth=2)  # type: ignore
plt.xlabel("Iteration")  # type: ignore
plt.ylabel("Binary Log Loss")  # type: ignore
plt.title("Binary Log Loss Over Training Iterations")  # type: ignore
plt.legend()  # type: ignore
plt.grid(alpha=0.3)  # type: ignore
plt.tight_layout()
plt.show()  # type: ignore

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)  # type: ignore
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=["No Diabetes", "Diabetes"]
)
disp.plot(cmap="Blues")  # type: ignore
plt.title(f"Recall: {recall_test_positive}")  # type: ignore
plt.tight_layout()
plt.show()  # type: ignore

## 8. Feature Importance

In [ ]:
fi = pd.Series(best_model.feature_importances_, index=X.columns)
fi_sorted = fi.sort_values(ascending=True)

plt.figure(figsize=(10, 8))  # type: ignore
fi_sorted.plot(kind="barh")
plt.title("Feature Importances Splits (LightGBM)")  # type: ignore
plt.xlabel("Importance")  # type: ignore
plt.tight_layout()
plt.show()  # type: ignore

In [ ]:
# Display gain-based feature importance locally
fi_gain = pd.Series(
    best_model.booster_.feature_importance(importance_type="gain"),  # type: ignore
    index=X.columns,
).sort_values(ascending=True)

plt.figure(figsize=(10, 8))  # type: ignore
fi_gain.plot(kind="barh")
plt.title("Feature Importances by Gain (LightGBM)")  # type: ignore
plt.xlabel("Gain")  # type: ignore
plt.tight_layout()
plt.show()  # type: ignore

In [ ]:
# Threshold optimization using selected inference model
y_proba = inference_model.predict_proba(X_val)[:, 1]

# Calculate PR curve
precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)
pr_auc = auc(recalls, precisions)

# Create figure with 2 subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Precision-Recall Curve ---
ax1 = axes[0]
ax1.plot(recalls, precisions, "b-", linewidth=2, label=f"PR Curve (AUC={pr_auc:.3f})")
ax1.fill_between(recalls, precisions, alpha=0.2)

# Mark key threshold points
target_recalls = [0.80, 0.75, 0.70, 0.50]
colors = ["red", "orange", "green", "purple"]

for target, color in zip(target_recalls, colors):
    idx = np.where(recalls[:-1] >= target)[0]
    if len(idx) > 0:
        i = idx[-1]
        thresh = thresholds[i]
        ax1.scatter(
            recalls[i],
            precisions[i],
            c=color,
            s=100,
            zorder=5,
            label=f"Recall={target:.0%} (thresh={thresh:.3f}, prec={precisions[i]:.1%})",
        )

# Baseline (random classifier)
baseline = y_val.mean()
ax1.axhline(
    y=baseline,
    color="gray",
    linestyle="--",
    label=f"Baseline (prevalence={baseline:.1%})",
)

ax1.set_xlabel("Recall (Sensitivity)", fontsize=12)
ax1.set_ylabel("Precision (PPV)", fontsize=12)
calibration_label = "Calibrated" if USE_CALIBRATION else "Uncalibrated"
ax1.set_title(f"Precision-Recall Curve ({calibration_label})", fontsize=14)
ax1.legend(loc="upper right", fontsize=9)
ax1.set_xlim([0, 1.02])
ax1.set_ylim([0, 1.02])
ax1.grid(True, alpha=0.3)

# --- Plot 2: Threshold vs Metrics ---
ax2 = axes[1]

# Calculate recall and precision at each threshold
thresh_range = np.linspace(0.05, 0.6, 100)
recall_at_thresh = []
precision_at_thresh = []
f1_at_thresh = []
flagged_pct = []

for t in thresh_range:
    y_pred = (y_proba >= t).astype(int)
    r = recall_score(y_val, y_pred, zero_division=0)
    p = precision_score(y_val, y_pred, zero_division=0)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    recall_at_thresh.append(r)
    precision_at_thresh.append(p)
    f1_at_thresh.append(f1)
    flagged_pct.append(y_pred.mean())

ax2.plot(thresh_range, recall_at_thresh, "b-", linewidth=2, label="Recall")
ax2.plot(thresh_range, precision_at_thresh, "r-", linewidth=2, label="Precision")
ax2.plot(thresh_range, f1_at_thresh, "g--", linewidth=2, label="F1 Score")
ax2.plot(thresh_range, flagged_pct, "k:", linewidth=2, label="% Flagged")

# Mark default 0.5 threshold
ax2.axvline(x=0.5, color="gray", linestyle="--", alpha=0.7, label="Default (0.5)")

# Mark optimal threshold for 80% recall
target_80_idx = np.argmin(np.abs(np.array(recall_at_thresh) - 0.80))
optimal_thresh = thresh_range[target_80_idx]
ax2.axvline(
    x=optimal_thresh,
    color="red",
    linestyle="--",
    alpha=0.7,
    label=f"80% Recall (thresh={optimal_thresh:.3f})",
)

ax2.set_xlabel("Decision Threshold", fontsize=12)
ax2.set_ylabel("Score", fontsize=12)
ax2.set_title(f"Metrics vs Decision Threshold ({calibration_label})", fontsize=14)
ax2.legend(loc="center right", fontsize=9)
ax2.set_xlim([0.05, 0.6])
ax2.set_ylim([0, 1.02])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("precision_recall_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Print Summary Table ---
print("\n" + "=" * 70)
print(f"THRESHOLD ANALYSIS SUMMARY ({calibration_label.upper()} PROBABILITIES)")
print("=" * 70)
print(f"{'Threshold':<12} {'Recall':<12} {'Precision':<12} {'F1':<12} {'Flagged':<12}")
print("-" * 70)

for thresh in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    y_pred = (y_proba >= thresh).astype(int)
    r = recall_score(y_val, y_pred, zero_division=0)
    p = precision_score(y_val, y_pred, zero_division=0)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    n_flagged = y_pred.sum()
    pct_flagged = y_pred.mean() * 100
    print(
        f"{thresh:<12.2f} {r:<12.1%} {p:<12.1%} {f1:<12.3f} {n_flagged} ({pct_flagged:.1f}%)"
    )

print("=" * 70)
print(
    f"\nValidation set: {len(y_val)} samples, {y_val.sum()} diabetes cases ({y_val.mean():.1%} prevalence)"
)

# Post Hoc - Threshold optimization

In [ ]:
# Final test evaluation using selected inference model
y_test_proba = inference_model.predict_proba(X_test)[:, 1]
y_pred_final = (y_test_proba >= optimal_thresh).astype(int)

### Apply final evaluation on test set

In [ ]:
r = recall_score(y_test, y_pred_final, zero_division=0)  # type: ignore
p = precision_score(y_test, y_pred_final, zero_division=0)  # type: ignore
f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0

In [ ]:
print(
    f"\nFinal chosen threshold: {optimal_thresh}"
    f"\n - Recall: {r:.2%}"
    f"\n - Precision: {p:.2%}"
    f"\n - F1 Score: {f1:.3f}"
)

In [ ]:
# Confusion matrix at final chosen threshold
cm_final = confusion_matrix(y_test, y_pred_final)  # type: ignore
disp_final = ConfusionMatrixDisplay(
    confusion_matrix=cm_final,
    display_labels=["No Diabetes", "Diabetes"],
)
disp_final.plot(cmap="Blues")  # type: ignore
plt.title(
    "Confusion Matrix at Final Threshold\n"
    f"(threshold={optimal_thresh:.3f}, recall={r:.2%}, precision={p:.2%})"
)  # type: ignore
plt.tight_layout()
plt.show()  # type: ignore

## 9. W&B Final Model Evaluation Logging

This section logs detailed evaluation metrics to Weights & Biases for the winning model.

In [ ]:
# Start a dedicated run for final model analysis
full_params = get_full_params(fixed_params, study.best_params)

final_run = wandb.init(
    project=WANDB_PROJECT,
    entity=ENTITY,
    name=f"{STUDY_NAME}-final-evaluation",
    job_type="evaluation",
    config={
        **full_params,
        "best_cv_ap": study.best_value,
        "chosen_threshold": optimal_thresh,
        "use_calibration": USE_CALIBRATION,
        "calibration_method": "isotonic" if USE_CALIBRATION else None,
    },
)

In [ ]:
# Log interactive PR curve
# W&B expects probabilities for both classes
y_probas_both = inference_model.predict_proba(X_test)

wandb.log({
    "pr_curve": wandb.plot.pr_curve(
        y_true=y_test.values, y_probas=y_probas_both, labels=["No Diabetes", "Diabetes"]
    )
})

In [ ]:
# Log confusion matrix
wandb.log({
    "confusion_matrix": wandb.plot.confusion_matrix(
        y_true=y_test.values,
        preds=y_pred_final,
        class_names=["No Diabetes", "Diabetes"],
    )
})

In [ ]:
# Log threshold analysis table
y_val_proba_final = inference_model.predict_proba(X_val)[:, 1]

threshold_data = []
for t in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    y_pred_t = (y_val_proba_final >= t).astype(int)
    threshold_data.append([
        t,
        recall_score(y_val, y_pred_t, zero_division=0),
        precision_score(y_val, y_pred_t, zero_division=0),
        f1_score(y_val, y_pred_t, zero_division=0),
        y_pred_t.mean(),  # proportion flagged
    ])

wandb.log({
    "threshold_analysis": wandb.Table(
        columns=["Threshold", "Recall", "Precision", "F1", "Pct_Flagged"],
        data=threshold_data,
    )
})

In [ ]:
# Log final summary metrics (appear in W&B run summary)
wandb.summary.update({
    # Test set performance
    "test_recall": recall_score(y_test, y_pred_final),
    "test_precision": precision_score(y_test, y_pred_final),
    "test_f1": f1_score(y_test, y_pred_final),
    # Model info
    "chosen_threshold": optimal_thresh,
    "best_cv_ap": study.best_value,
    "n_iterations_used": best_model.best_iteration_,
    "use_calibration": USE_CALIBRATION,
    # Data info
    "test_size": len(y_test),
    "test_prevalence": y_test.mean(),
})

In [ ]:
# Log feature importance (gain-based)
fi = pd.Series(
    best_model.booster_.feature_importance(importance_type="gain"), index=X.columns
).sort_values(ascending=False)

wandb.log({
    "feature_importance": wandb.Table(
        columns=["Feature", "Gain_Importance"],
        data=[[feat, imp] for feat, imp in fi.items()],
    )
})

# Also log as bar chart (top 10)
wandb.log({
    "feature_importance_chart": wandb.plot.bar(
        wandb.Table(
            columns=["Feature", "Importance"],
            data=[[feat, imp] for feat, imp in fi.head(10).items()],
        ),
        "Feature",
        "Importance",
        title="Top 10 Features (Gain)",
    )
})

In [ ]:
# Close the W&B run
wandb.finish()

---